In [2]:
import pandas as pd

In [3]:
df = pd.read_excel("../plan_de_compras_2025.xlsx")

In [4]:
display(df.head())

,Unidad de Compra,ID Proyecto,Tipo Proyecto,Estado Proyecto,Código presupuestario,Nombre Proyecto,Descripción Proyecto,Cantidad de Ítems,Nombre ítem,Tipo compra,...,Cargo responsable,Teléfono responsable,Correo responsable,Fecha de Inicio Compra,Fecha Publicación PAC 2025,Cantidad OC Asociadas Ítem 2025,OC Asociada Item 2025,Item De Arraste,Monto De Arrastre,Meses envio OC de Arraste
0,Ministerio de Vivienda y Urbanismo(766),587-259-PC22,Proyecto estratégico,Publicado,2207-2209-33,Servicio arriendo de una plataforma de gestión...,Servicio de arriendo de una plataforma de gest...,2,Servicio arriendo de una plataforma de gestión...,1,...,Coordinadora de Tecnologías Informáticas | Sis...,2 -2901 1198-,mzunigas@minvu.cl,2022-11-01,2022-11-08,0,NaN,1,20000000,2025-01-01
1,SEREMI MINVU V REGION,632-28-PC22,Proyecto operacional,Publicado,2206-2206001,Servicio de Mantención y Reparaciones Menores ...,Convenio de Suministro para el Servicio de Man...,1,Mantenimiento y Reparación de Edificaciones,1,...,Encargado Sección Administración y Finanzas,32-2186802-,agardaix@minvu.cl,2022-02-01,2022-11-08,1,632-7-SE22,1,4500000,2025-01-01
2,SEREMI MINVU V REGION,632-32-PC22,Proyecto operacional,Publicado,2204001,Servicios de Impresión-Seremi Valparaiso,Servicios de Impresión de Materiales SEREMI y ...,1,Materiales de Oficina,1,...,Encargado Sección Administración y Finanzas,32-2186802-,agardaix@minvu.cl,2022-08-01,2022-11-08,0,NaN,1,3825000,2025-01-01
3,SEREMI MINVU V REGION,632-34-PC22,Proyecto operacional,Publicado,2209006,Servicio de Arriendo de Impresoras Multifuncio...,Servicio de Arriendo de Impresoras Multifuncio...,2,Arriendo de Equipos Informáticos,1,...,Encargado Sección Administración y Finanzas,32-2186802-,agardaix@minvu.cl,2022-08-01,2022-11-08,1,632-13-CC23,1,4500000,2025-01-01
4,SEREMI MINVU V REGION,632-41-PC22,Proyecto operacional,Publicado,2205005,Servicio de Telefonía Fija-Seremi Valparaiso,Servicio de Telefonía Fija SEREMI,2,Telefonía Fija,1,...,Encargado Sección Administración y Finanzas,322186802,agardaix@minvu.cl,2022-12-01,2022-11-08,0,NaN,1,26400000,2025-01-01


In [5]:
# Copia para no modificar el DataFrame original
df_limpio = df.copy()

In [6]:
# Normalizar nombres de columnas
df_limpio.columns = (
    df_limpio.columns.astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", "_", regex=True)
)


In [7]:
# Limpiar espacios y convertir textos vacíos a valores nulos
columnas_texto = df_limpio.select_dtypes(include="object").columns

for columna in columnas_texto:
    df_limpio[columna] = (
        df_limpio[columna]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )


In [8]:
# Eliminar filas y columnas completamente vacías
df_limpio = df_limpio.dropna(how="all")
df_limpio = df_limpio.dropna(axis=1, how="all")

In [9]:
# Reporte de nulos para revisar antes de imputar o eliminar
reporte_nulos = (
    df_limpio.isna()
    .sum()
    .to_frame("cantidad_nulos")
    .assign(porcentaje_nulos=lambda x: (x["cantidad_nulos"] / len(df_limpio) * 100).round(2))
    .sort_values("cantidad_nulos", ascending=False)
)

display(reporte_nulos[reporte_nulos["cantidad_nulos"] > 0])

,cantidad_nulos,porcentaje_nulos
meses_envio_oc_de_arraste,648,77.98
oc_asociada_item_2025,123,14.80
código_presupuestario,1,0.12


In [10]:
filas_antes = len(df_limpio)

df_sin_nulos = df_limpio.dropna().copy()

filas_eliminadas = filas_antes - len(df_sin_nulos)

print(f"Filas antes: {filas_antes}")
print(f"Filas eliminadas por nulos: {filas_eliminadas}")
print(f"Filas finales: {len(df_sin_nulos)}")
print(f"Nulos restantes: {df_sin_nulos.isna().sum().sum()}")

Filas antes: 831
Filas eliminadas por nulos: 717
Filas finales: 114
Nulos restantes: 0


In [11]:
df_sin_nulos.to_excel(
    "../plan_de_compras_2025_sin_nulos.xlsx",
    index=False,
    sheet_name="datos_sin_nulos"
)